# Code as Policies with CaP-X

[CaP-X](https://github.com/capgym/cap-x) (*Code-as-Policies eXtended*) is a framework for benchmarking and improving coding agents for robot manipulation. Instead of asking a model what the arm should do next, it asks the model to **write a Python program** that solves the task by composing perception and control primitives, runs that program in simulation, and scores whether the task actually got done.

The robot is a Franka Panda again, this time in a Robosuite/MuJoCo simulation. The model that writes the program is served locally, so the reasoning stays on your Radeon GPU.

## Goals

* Understand how a code-generating agent controls a robot, and how that differs from the tool-calling agent in the previous notebook
* Configure CaP-X against your own OpenAI-compatible model server
* Read the program the model writes, and the prompt that produced it
* Measure a model by its success rate over repeated trials instead of a single demo

## Two ways to put an LLM on a robot

Notebook 03 and this notebook attempt to solve the same problem of robot manipulation through using natural language as an input, and planned motion of the arm as an output. Their approach differs with the local model's responsibility: **calling tools vs generating code.**

| | Notebook 03 - RAI | This notebook - CaP-X |
| --- | --- | --- |
| What the model emits | one tool call at a time | one Python program, up front |
| Where the model sits | inside the control loop | upstream of it |
| Model calls per task | one per step, dozens of them | one |
| What drives the robot | a LangGraph loop | CPython executing the generated code |
| Perception | a ROS 2 service, invoked as a tool | OWLv2, SAM2 and Contact-GraspNet, called as functions from inside the program |
| How you judge it | watch the arm | reward over N seeded trials, i.e. a success rate |
| A failure looks like | a tool call you can read | a traceback, or a program that runs cleanly and still misses |

The main tradeoff is: adaptability against efficiency. RAI can attempt any task with no preparation, but the model has to reason from scratch every time, so the hundredth run of a task costs the same as the first. CaP-X generates a program per task, and that program can then be re-run without a model at all; yet a wrong read of the scene spoils the whole episode, and every new task needs another program.

## Serve the model locally

CaP-X is compatible with plain OpenAI chat completions, therefore any server that can communicate in the protocol will do. We will use the same Lemonade server as notebooks 02 and 03, on the same port.

`ensure_lemonade` will starts `lemond` if it is not already up and loads the model onto the iGPU, hence whether you left the server running after the earlier notebooks won't affect the result for this notebook.

In [ ]:
import sys

sys.path.insert(0, "/ryzers")

from capx_demo import (
    benchmark_scenarios,
    ensure_lemonade,
    quiet,
    show_trial_grid,
    show_video,
)

MODEL = "Gemma-4-E2B-it-GGUF"
ensure_lemonade(MODEL)

## Start the perception services and configure CaP-X

Four servers sit behind the primitives the generated program may call. Grounding a noun phrase such as `"red cube"` into the pixels that are the cube takes two of them: OWLv2 turns the phrase into a bounding box, and SAM2 turns that box into a mask. Contact-GraspNet then turns that mask plus depth into ranked 6-DoF grasp poses, and PyRoKi solves the inverse kinematics. `FrankaControlApi` wraps them into the five functions the model is allowed to use.

Upstream CaP-X does the grounding in one model, SAM3, which takes the phrase and returns masks directly. Its weights are gated on Hugging Face, so building this image would need an account with approved access and a token. OWLv2 and SAM2 are both public and split the same job in two, which is why nothing here asks you to log in.

The cell below is what `capx/envs/launch.py` does before its first trial, one call at a time. `LaunchArgs` is the dataclass the CLI parses its flags into, so filling one in by hand configures the framework exactly as a terminal run would, and `model` and `server_url` are the whole coupling between CaP-X and whatever is doing the reasoning. Point them at any OpenAI-compatible chat-completions endpoint and nothing else in this notebook changes.

In [ ]:
from capx.envs.configs.instantiate import instantiate
from capx.envs.launch import LaunchArgs
from capx.envs.runner import _start_api_servers
from capx.utils.launch_utils import _load_config

SERVER_URL = "http://localhost:13305/api/v1/chat/completions"
CONFIG_PATH = "env_configs/cube_stack/franka_robosuite_cube_stack_sam2.yaml"

TEMPERATURE = 0.2
MAX_TOKENS = 16384

args = LaunchArgs(
    config_path=CONFIG_PATH,
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

# Bringing the stack up is loud and none of it is about CaP-X, so `quiet` sends
# it to /tmp/capx_notebook/setup.log. Read that file if a server misbehaves.
with quiet() as log:
    # The config declares the servers it needs under `api_servers:`, and
    # _load_config hands them back next to the factory for the env they serve.
    env_factory, config, api_servers = _load_config(args)

    # OWLv2, SAM2, Contact-GraspNet and PyRoKi, one process each.
    servers = _start_api_servers(api_servers, 900.0)

    # instantiate walks the _target_ entries in the YAML and builds the object
    # graph: the code-execution env, the Robosuite task under it, and the
    # FrankaControlApi holding the clients for the servers above.
    env = instantiate(env_factory)
    api = next(iter(env._apis.values()))

    # reset re-samples where the cubes land on the table
    obs, _ = env.reset(options={"trial": 0}, seed=0)

for server in api_servers or []:
    print(f"API server on {server.get('host', '127.0.0.1')}:{server.get('port')} is ready")
print("ready:", type(env).__name__, "|", type(api).__name__, "| log:", log)

## Generate a program for the task

`ModelQueryArgs` carries the same values as `LaunchArgs`, and `query_model` is the chat-completions call underneath, so this is one HTTP round trip to the server you configured above. `obs["full_prompt"]` is the two-message conversation the environment built for the scene it reset to.

The program the model writes is a short list of **tool** calls: `get_object_pose`, `sample_grasp_pose`, `goto_pose` and the rest. Under each of those tools is the **lower-level** code that actually talks to the perception and IK servers, which the cell prints below the program so you can see both halves at once.

In [ ]:
import inspect

from capx.llm.client import ModelQueryArgs, query_model
from capx.utils.launch_utils import _extract_code

query_args = ModelQueryArgs(
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

response = query_model(query_args, obs["full_prompt"])

blocks = _extract_code(response["content"])
assert blocks, f"no code in the reply - raise MAX_TOKENS?\n{response['content'][-500:]}"

program = blocks[0]

print("======================== Tools ========================")
print(program)

# The tools the program actually called, shown from the code side: the lower
# level that reaches the perception and IK servers, which the model never sees.
print("\n===================== Lower level =====================")
primitives = api.functions()
for name, fn in primitives.items():
    if f"{name}(" not in program:
        continue
    print(f"# ---- {name} ----")
    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError):
        print(f"({name}: source unavailable)\n")

## Run program

**The generated code is the policy.**

If it fails, the output will display how. `No detections` means the noun phrase did not ground, and a clean run below `1.0` means the geometry was wrong.

A traceback usually means the model got Python wrong, which is the most common failure for small models, but read it before blaming the program. `argmax of an empty sequence` inside `sample_grasp_pose` is the perception stack rather than the model: the object was grounded but the mask caught only its top face, so the depth extent came back near zero, and Contact-GraspNet cannot fit a grasp to a plane. Compare the `Object extent` line for the object against its real height, roughly 4 cm for a cube. That one is a property of the layout, so a different `seed=` in the reset above is enough to move past it.

In [4]:
env.enable_video_capture(True, clear=True)

# The sandbox echoes the program's prints as they happen and the perception
# servers log every request they answer. Both go to the log file, and the same
# program output comes back captured in info["stdout"], printed below.
with quiet():
    _, reward, terminated, _, info = env.step(program)

print(f"reward {reward:.3f} | solved {info['task_completed']} | terminated {terminated}")
if info["stdout"].strip():
    print("\n----- program stdout (tail) -----\n" + info["stdout"][-2000:])
if info["sandbox_rc"]:
    print("\n----- traceback (tail) -----\n" + info["stderr"][-2000:])

show_video(env)

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/usr/local/lib/python3.12/dist-packages/_rocm_sdk_libraries_gfx1151/share/miopen/db/gfx1151_20.HIP.fdb.txt"
/ryzers/cap-x/capx/third_party/sam3/sam3/model/vitdet.py:504: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:309.)
  x = F.scaled_dot_product_attention(q, k, v)
/ryzers/cap-x/capx/third_party/sam3/sam3/model/vitdet.py:504: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:360.)
  x = F.scaled_dot_product_attention(q, k, v)


a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_nperblock_container_{5376, 512}
a_grid_desc_m_ak_container_{5376, 1024}
b_grid_desc_n_bk_container_{512, 1024}
e_grid_desc_mblock_mperblock_nblock_npe

MIOpen(HIP): Warning [ParseAndLoadDb] File is unreadable: "/usr/local/lib/python3.12/dist-packages/_rocm_sdk_libraries_gfx1151/share/miopen/db/gfx1151_20.HIP.fdb.txt"


Generated 2 grasps for object 1
INFO:     127.0.0.1:50466 - "POST /plan HTTP/1.1" 200 OK
Grasp sample position for red cube: [ 0.48996223  0.10227364 -0.09508278]
Grasp sample quaternion wxyz for red cube: [-0.17585823  0.22486646  0.95650228  0.06010337]
INFO:     127.0.0.1:60844 - "POST /segment HTTP/1.1" 200 OK
Object position for red cube: [ 0.49833099  0.0874119  -0.08493799]
Object quaternion wxyz for red cube: [-0.46784045  0.07469621  0.87915811  0.05125232]
Object extent for red cube: [0.05584644 0.04117704 0.03306428]


2026-08-17 22:04:16.784 | INFO     | jaxls._problem:analyze:201 - Building optimization problem with 2 terms and 1 variables: 1 costs, 0 eq_zero, 1 leq_zero, 0 geq_zero
2026-08-17 22:04:16.799 | INFO     | jaxls._problem:analyze:320 - Vectorizing constraint group with 1 constraints (constraint_leq_zero), 1 variables each: augmented_limit_residual
2026-08-17 22:04:17.140 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: _pose_cost_analytical_jac


INFO:     127.0.0.1:42810 - "POST /ik HTTP/1.1" 200 OK


2026-08-17 22:04:18.784 | INFO     | jaxls._problem:analyze:201 - Building optimization problem with 3 terms and 1 variables: 3 costs, 0 eq_zero, 0 leq_zero, 0 geq_zero
2026-08-17 22:04:18.803 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: _pose_cost_analytical_jac
2026-08-17 22:04:18.809 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: limit_residual
2026-08-17 22:04:18.815 | INFO     | jaxls._problem:analyze:328 - Vectorizing group with 1 costs, 1 variables each: limit_velocity_cost


INFO:     127.0.0.1:42824 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42832 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42836 - "POST /ik HTTP/1.1" 200 OK
INFO:     127.0.0.1:42852 - "POST /ik HTTP/1.1" 200 OK
reward 1.000 | solved True | terminated True

----- program stdout (tail) -----
Object position for green cube: [ 0.59241932  0.09679693 -0.08111711]
Object quaternion wxyz for green cube: [ 0.37774157  0.42647165 -0.73584845 -0.36600588]
Object extent for green cube: [0.07756438 0.07275727 0.06082484]
Grasp sample position for red cube: [ 0.48996223  0.10227364 -0.09508278]
Grasp sample quaternion wxyz for red cube: [-0.17585823  0.22486646  0.95650228  0.06010337]
Object position for red cube: [ 0.49833099  0.0874119  -0.08493799]
Object quaternion wxyz for red cube: [-0.46784045  0.07469621  0.87915811  0.05125232]
Object extent for red cube: [0.05584644 0.04117704 0.03306428]

Saved interaction video to /tmp/capx_notebook/video_notebook_run.mp4 (34 frames)


'/tmp/capx_notebook/video_notebook_run.mp4'

## Results across different tasks

Cube stacking is one task, and the model writes a different program for each. `benchmark_scenarios` runs one episode of six different Robosuite tasks - cube stack, cube restack, cube lift, nut assembly, spill wipe and two arm handover - each grounded with the same OWLv2 and SAM2 servers the cell above already started, so nothing new has to start. Every task writes its own directory with the generated program, the model response, the prompt and an MP4, and a task that fails to run is recorded so the rest continue.

Two arm handover is the one that asks for something the others do not: two Panda arms, where the program has to pass an object from one to the other rather than drive a single gripper. Watch whether the model keeps track of which arm it is commanding.

The cell below the results lays the six episodes out side by side, captioned by task, so you can see where the model turns an instruction into a working program and where it does not.

In [ ]:
scenarios = benchmark_scenarios(
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

## The six episodes, side by side

In [ ]:
show_trial_grid(scenarios, width=240)

## Testing a more powerful model

Gemma-4-E2B is a 2B model, and one-shot manipulation programs are near the edge of what it can do, as the table above shows. `Ministral-3-14B-Instruct` is roughly seven times larger and is tuned for instruction following and tool use, hence, it is a good second data point. Its Q4_K_M weights are about 8 GB against around 4 GB for the Gemma-4-E2B.

The run below portrays the same six tasks at the same seeds, to serve as comparison success rates against Gemma above. Expect it to take considerably longer, since the model is bigger and every program is a fresh generation.

Ministral is not one of Lemonade's built-in names, so the image registers it as a `user.` model pinned to a single `.gguf` file. That prefix is why the name below is not simply the repository name.

In [ ]:
BIG_MODEL = "user.Ministral-3-14B-Instruct-2512-GGUF"
ensure_lemonade(BIG_MODEL)

scenarios_big = benchmark_scenarios(
    model=BIG_MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

In [ ]:
show_trial_grid(scenarios_big, width=240)

## Where the motion happens

`goto_pose` is the one primitive that moves the arm, and it does three things worth knowing about. Every pose in this API is a gripper-tip pose, so it adds the 10.7 cm offset to the `panda_hand` link before solving, or the arm drives into the table. A non-zero `z_approach` makes it solve and execute *twice*, once at a standoff along the gripper's local -Z and once at the target, which is why the task prompt insists on `z_approach=0.1`. And it hands the previous joint configuration back to the solver, which then penalises joint velocity, so the elbow does not flip across the table between waypoints.

The solve is nonlinear least squares over the joint vector, so it always returns something: an unreachable target comes back as the closest configuration it could find, with no error and no flag. When a program fails for no visible reason, suspect that first.

In [6]:
import inspect

print(inspect.getsource(type(api).goto_pose))

    def goto_pose(
        self, position: np.ndarray, quaternion_wxyz: np.ndarray, z_approach: float = 0.0
    ) -> None:
        """Go to pose using Inverse Kinematics.
        There is no need to call a second goto_pose with the same position and quaternion_wxyz after calling it with z_approach.
        Args:
            position: (3,) XYZ in meters.
            quaternion_wxyz: (4,) WXYZ unit quaternion.
            z_approach: (float) Z-axis distance offset for goto_pose insertion approach motion. Will first arrive at position + z_approach meters in Z-axis before moving to the requested pose. Useful for more precise grasp approaches. Default is 0.0.
        Returns:
            None
        """
        pos_str = np.array2string(np.asarray(position), precision=4)
        approach_info = f" (z_approach={z_approach:.3f})" if z_approach != 0.0 else ""
        self._log_step("goto_pose", f"Moving to position {pos_str}{approach_info} …")

        pos = np.asarray(position, dtype=np.floa

## System Prompt

Two messages are used as prompts for the api calls, the ones handed to `query_model` above. The system message is one sentence, and the user message is the task description followed by an `APIs:` section that is not written by hand anywhere: CaP-X renders it from the signature and docstring of every primitive the program may call. So the docstrings *are* the prompt, and it cannot drift out of date with the code.

In [7]:
system, user = obs["full_prompt"]

print(system["content"])
print(user["content"][0]["text"])

You are a helpful assistant that generates Python code to directly solve the task.

You are controlling a Franka Emika robot with the API described below.
Goal: Pick up the red cube and gently stack it on top of the green cube, then release it.

Key rules:
- The extent from get_object_pose(..., return_bbox_extent=True) is the FULL side length. Use extent[2]/2 for half-height.
- For placement orientation, reuse the grasp quaternion from sample_grasp_pose. Do NOT use the quaternion from get_object_pose (it is unreliable for orientation).
- Always use z_approach=0.1 when approaching an object for grasping or placing.
- After grasping, lift the cube to a safe height (at least +0.2m in Z) before moving laterally to the placement location.
- The stacking height formula is: place_z = green_center_z + green_extent[2]/2 + red_extent[2]/2
- Nothing should be dropped from a height. Always approach with z_approach for controlled descent.

Write ONLY executable Python code (no code fences). Import 

> **Note:** nothing in that prompt describes the scene. There is no image, no object coordinates and no joint state, so the model writes the program blind, and every fact about the actual arrangement is obtained at runtime by the program itself, through the perception servers.

## Key Takeaways

Now you know:
- How a Code-as-Policies agent differs from a tool-calling agent: one program written up front instead of one decision per step, and a success rate instead of a demo
- How to configure CaP-X against any OpenAI-compatible server, with `LaunchArgs` and `ModelQueryArgs` carrying the same `--model` and `--server-url` the CLI takes
- How the generated program gets from a noun phrase to motion: grounding into a mask, mask plus depth into a grasp pose, pose into joint angles through IK
- That the prompt is generated from the primitives' docstrings, so the API and its documentation cannot drift apart
- How to benchmark a locally served model across several different tasks and read the result

## What to Try Next

- Point `SERVER_URL` at a llama.cpp, Ollama or vLLM server, or at the OpenRouter proxy, and rerun from that cell down to understand how different models perform and how to use different backends
- Run `benchmark_scenarios(..., oracle=True)` to see every task succeed with no model in the loop, and use it as your control whenever a result looks wrong
- Edit a docstring in `FrankaControlApi` and watch the generated program change; you are editing the prompt. `/ryzers/cap-x` is an editable install, so the change lands on the next kernel restart
- Raise `temperature` and generate a few programs for the same scene, to see how much of the policy is the model guessing

## References

* [CaP-X](https://github.com/capgym/cap-x)
* [OWLv2](https://huggingface.co/google/owlv2-large-patch14-ensemble) · [SAM 2.1](https://huggingface.co/facebook/sam2.1-hiera-large) · [Contact-GraspNet](https://github.com/NVlabs/contact_graspnet) · [PyRoKi](https://github.com/chungmin99/pyroki)
* [Robosuite](https://github.com/ARISE-Initiative/robosuite)
* [Lemonade](https://lemonade-server.ai/)